Dans ce pipeline , on va utiliser la RS pour retrouver une formule qu’on connaît déjà à savoir:
$$E^2 = P_x^{2} + P_y^{2}+ P_z^{2}+ mass^2 $$

Nons essayons de répondre à la question suivante :La RS peut-elle retrouver une loi physique connue à partir de données bruitées ?

In [ ]:
!pip install pyhepmc -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.6/625.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.4/494.4 kB 29.2 MB/s eta 0:00:00


In [ ]:
import pyhepmc
import pandas as pd
import numpy as np


In [ ]:
reader = pyhepmc.open("/content/HEPMC.43646133._000001.hepmc")

events = []


In [ ]:
for evt in reader:
    event_id = evt.event_number

    for p in evt.particles:
        if p.status == 1:  # particules finales
            events.append({
                "event_id": event_id,
                "pid": p.pid,
                "px": p.momentum.px,
                "py": p.momentum.py,
                "pz": p.momentum.pz,
                "E":  p.momentum.e,
                "mass": p.generated_mass
            })

df = pd.DataFrame(events)


In [ ]:
df.head()



,event_id,pid,px,py,pz,E,mass
0,1,-211,-111.128639,56.038723,1.907730e+04,1.907822e+04,139.570007
1,1,2112,0.295795,-21.378218,1.180517e+06,1.180518e+06,939.570007
2,1,-211,-514.510132,430.511658,9.006374e+04,9.006635e+04,139.570007
3,1,2112,-524.820435,852.246521,2.262901e+04,2.267061e+04,939.570007
4,1,-2212,0.360639,161.869766,1.196875e+04,1.200656e+04,938.270020


In [ ]:
df.tail()


,event_id,pid,px,py,pz,E,mass
211535,577,22,-72.129692,261.683167,358.075165,449.331276,0.000000
211536,577,22,-223.396469,271.909851,546.435791,649.948477,0.000008
211537,577,22,-75.457382,4.186639,-64.139023,99.121939,0.000000
211538,577,22,18.923904,-25.391039,-209.816879,212.193171,0.000000
211539,577,130,3388.228760,1362.995480,-4839.871580,6083.569960,497.609985


In [ ]:
df.shape


(211540, 7)

In [ ]:
df.columns


Index(['event_id', 'pid', 'px', 'py', 'pz', 'E', 'mass'], dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211540 entries, 0 to 211539
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   event_id  211540 non-null  int64  
 1   pid       211540 non-null  int64  
 2   px        211540 non-null  float64
 3   py        211540 non-null  float64
 4   pz        211540 non-null  float64
 5   E         211540 non-null  float64
 6   mass      211540 non-null  float64
dtypes: float64(5), int64(2)
memory usage: 11.3 MB


In [ ]:
df.isna().any().sum()


np.int64(0)

In [ ]:
df.duplicated().sum()


np.int64(0)

In [ ]:
df["mass"].unique()
# On voit que la masse varie , ce qui signifie que df contient plusieurs particules

array([ 1.39570007e+02,  9.39570007e+02,  9.38270020e+02,  0.00000000e+00,
        4.93679993e+02,  4.97609985e+02,  1.11568005e+03,  1.39570175e+02,
        4.97614014e+02,  1.84678242e-06, -1.07895930e-05,  1.07895930e-05,
       -7.62939453e-06,  5.10998905e-01,  1.18937000e+03,  5.10999978e-01,
        1.19744995e+03,  1.05658371e+02,  2.15791861e-05, -1.52587891e-05,
       -2.64289974e-05,  2.64289974e-05,  4.93677002e+02, -6.10351562e-05,
       -4.31583721e-05,  3.05175781e-05, -5.98019978e-04, -1.72633489e-04,
        9.76562500e-04, -1.38106791e-03, -2.11431980e-04, -7.47524973e-05,
       -5.39479652e-06,  2.53038397e-05,  3.73762487e-05,  1.22070312e-04,
        5.10995150e-01,  3.47953930e-04,  1.72633489e-04, -4.88281250e-04,
        1.31485999e+03,  2.35153730e-05,  5.10998964e-01,  1.93010113e-04,
       -6.90533954e-04,  1.32144987e-05,  6.10351562e-05,  3.81469727e-06,
       -2.41262642e-05, -1.32144987e-05,  4.31583721e-05, -9.76562500e-04,
        3.45266977e-04,  

La relation qu'on cherche à redécouvrir est:
$$E^2 = P_x^{2} + P_y^{2}+ P_z^{2}+ mass^2 $$

Cette loi n’est vraie que pour une particule donnée, avec une masse fixe.
Si on mélange les pions ,neutrons , protons alors la mass n'est plus constant et la loi devient non fermée et la RS n'a aucune loi simple à découvrir.

Pour éviter cela , on travaillera avec les pions chargés ($𝜋$, PDG ID ±211) qui ont été isolés afin de garantir l’unicité de la masse et d’analyser la relation relativiste énergie–moment pour une espèce de particule donnée.

In [ ]:
df_pi = df[df.pid.abs() == 211].copy()


In [ ]:
df_pi

,event_id,pid,px,py,pz,E,mass
0,1,-211,-111.128639,56.038723,19077.298800,19078.215300,139.570007
2,1,-211,-514.510132,430.511658,90063.742200,90066.348900,139.570007
7,1,211,-217.619705,-291.487335,10678.606400,10685.711900,139.570007
8,1,-211,400.164764,285.474335,4009.708010,4042.136260,139.570007
9,1,211,-264.389435,-207.600571,10372.449200,10378.833400,139.570007
...,...,...,...,...,...,...,...
30906,84,-211,3.448694,-198.698502,688.547729,730.116943,139.570007
30907,84,-211,-584.771790,228.055344,3801.772710,3855.764880,139.570007
30908,84,211,-2783.829830,-767.590881,10452.445300,10844.906500,139.570007
30909,84,211,-1293.692140,-241.722336,3415.350830,3662.809050,139.570007


In [ ]:
df_pi.shape


(11555, 7)

In [ ]:
df_pi.pid.unique()


array([-211,  211])

In [ ]:
df_pi.describe()


,event_id,pid,px,py,pz,E,mass
count,11555.000000,11555.000000,11555.000000,11555.000000,1.155500e+04,1.155500e+04,11555.000000
mean,42.821549,1.807789,18.509547,10.317902,-6.372133e+02,3.304775e+04,139.570009
std,24.158401,211.001386,866.997287,1019.012823,1.343249e+05,1.302042e+05,0.000019
min,1.000000,-211.000000,-22309.115200,-38074.183600,-3.996468e+06,1.411702e+02,139.570007
25%,23.000000,-211.000000,-233.677032,-212.578453,-3.476471e+03,9.333682e+02,139.570007
50%,42.000000,211.000000,0.713756,7.644739,-5.930710e+01,3.242311e+03,139.570007
75%,63.000000,211.000000,231.221710,240.197854,2.713084e+03,1.575333e+04,139.570007
max,84.000000,211.000000,24547.919900,25638.609400,3.302195e+06,3.996468e+06,139.570175


Avant toute recherche de loi par régression symbolique, nous avons vérifié la cohérence relativiste des données en testant la relation énergie–impulsion.

Cette étape garantit que la RS opère sur un jeu de données physiquement valide, et permet d’interpréter ses résultats comme une véritable redécouverte ou une extension de lois physiques.

In [ ]:
import numpy as np

df_pi["p2"] = df_pi.px**2 + df_pi.py**2 + df_pi.pz**2
df_pi["rel_residual"] = df_pi.E**2 - (df_pi.p2 + df_pi.mass**2)

df_pi["rel_residual"].describe()


,rel_residual
count,11555.000000
mean,5.555288
std,740.010929
min,-22264.403076
25%,-0.017198
50%,0.000105
75%,0.017179
max,24624.005859


On vient de calculer :
$$rel_residuel=E^2 - P_x^{2} + P_y^{2}+ P_z^{2}+ mass^2 $$
L’analyse du résidu relativiste montre que les données respectent la relation énergie–impulsion avec une dispersion négligeable. Cette validation préalable garantit que la régression symbolique opère sur un jeu de données physiquement cohérent, permettant une interprétation fiable des lois redécouvertes ou de l’absence de structures supplémentaires.

In [ ]:
pip install gplearn -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.2 MB/s eta 0:00:00


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
X = df_pi[["px", "py", "pz"]].values
y = df_pi["E"].values


In [ ]:
X = df_pi[["px", "py", "pz","mass"]].values
y = df_pi["E"].values


In [ ]:
from gplearn.genetic import SymbolicRegressor
sr = SymbolicRegressor(
    population_size=3000,
    generations=25,
    stopping_criteria=1e-6,
    p_crossover=0.7,
    p_subtree_mutation=0.1,
    p_hoist_mutation=0.05,
    p_point_mutation=0.1,
    max_samples=0.9,
    verbose=1,
    parsimony_coefficient=0.05,
    random_state=42,
    function_set=['add', 'mul', 'sqrt']
)

In [ ]:
sr.fit(X,y)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    15.14      6.79748e+61       10          151.537          122.834      2.31m
   1    13.79      2.24433e+18       12          144.037          171.408      1.74m
   2    18.35      8.36899e+22       32          134.081          131.097      2.10m
   3    11.48      1.87861e+17       28          77.6252          63.6889      1.67m
   4    13.28      1.41252e+27       32          62.1381          60.2801      1.51m
   5    23.24      2.48844e+12       20          8.17963          7.47863      2.09m
   6    30.40       4.1732e+08       20          8.07325          8.43559      1.62m
   7    32.38      2.45021e+08       23          7.47627          7.36618      2.15m
   8    32.47      2.56445e+16       36          6.96003            6.418  

SymbolicRegressor(function_set=['add', 'mul', 'sqrt'], generations=25,
                  max_samples=0.9, p_crossover=0.7, p_hoist_mutation=0.05,
                  p_point_mutation=0.1, p_subtree_mutation=0.1,
                  parsimony_coefficient=0.05, population_size=3000,
                  random_state=42, stopping_criteria=1e-06, verbose=1)

La RS  a redécouvert exactement la relation de dispersion relativiste,sans que tu lui imposes la forme
et uniquement à partir des données:
$$
E_{\mathrm{RS}} = \sqrt{X_0^2 + X_1^2 + X_2^2 + X_3^2}
$$
avec :

$$
\begin{aligned}
X_0 &= p_x \\
X_1 &= p_y \\
X_2 &= p_z \\
X_3 &= m
\end{aligned}
$$


La RS :

 - ne connaît ni la relativité

- ni les invariants

- ni la métrique de Minkowski

- ni la notion de masse au repos

Elle :

- teste des millions d’expressions

- pénalise la complexité

- sélectionne celle qui explique les données au moindre coût

👉 Le fait que la loi exacte sorte est une validation méthodologique.